# Iris Prediction

Iris Prediction Network

Main Sources:
 
  [Kaggle Dataset](https://www.kaggle.com/datasets/arshid/iris-flower-dataset/data), 
  [Pytorch Basic Layout](https://docs.pytorch.org/tutorials/beginner/basics/quickstart_tutorial.html), 
  [Tensor Board for Visualising](https://docs.pytorch.org/tutorials/intermediate/tensorboard_tutorial.html)

## Setup: importing Packages + checking for a GPU

In [150]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.tensorboard import SummaryWriter

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using: {device}")

Using: mps


## Defining the neural network
Main things I changed from the reference:
- Linear : 4 &rarr; 10 &rarr; 5 &rarr; 3 (Random numbers, not much thought beyond that)
- ReLU &rarr; SiLU (I read somewhere that its better, more smooth)

In [151]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear_silu_stack = nn.Sequential(
            nn.Linear(4, 10),
            nn.SiLU(),
            nn.Linear(10, 5),
            nn.SiLU(),
            nn.Linear(5, 3)
        )

    def forward(self, x):
        logits = self.linear_silu_stack(x)
        return logits
    
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (linear_silu_stack): Sequential(
    (0): Linear(in_features=4, out_features=10, bias=True)
    (1): SiLU()
    (2): Linear(in_features=10, out_features=5, bias=True)
    (3): SiLU()
    (4): Linear(in_features=5, out_features=3, bias=True)
  )
)


## Importing Data, changing variety to numbers, splitting into traindata and testdata

In [152]:
data_path = './Data/IRIS.csv'

df = pd.read_csv(data_path)

df["species"] = df["species"].map({
    "Iris-setosa": 0,
    "Iris-versicolor": 1,
    "Iris-virginica": 2,
})

df_tensor = torch.from_numpy(df.values).float()
X = df_tensor[:, :-1]
y = df_tensor[:, -1].long()

dataset = torch.utils.data.TensorDataset(X, y)
traindata, testdata = torch.utils.data.random_split(dataset, [0.80, 0.20])

print(X.size(), y.size(), len(traindata), len(testdata))

torch.Size([150, 4]) torch.Size([150]) 120 30


## Batching

In [153]:
batch_size = 10 # Chose a small batch because the dataset is small
train_loader = torch.utils.data.DataLoader(traindata, batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(testdata, batch_size, shuffle=False)

## Learning
- Changed the optimisation model from SGD to AdamW (cause I read its better)

In [154]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001) # learning rate adjusted because of the small dataset

def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 3 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

epochs = 200
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_loader, model, loss_fn, optimizer)
    test(test_loader, model, loss_fn)

print("Done!")

torch.save(model.state_dict(), "./Data/model.pth")
print("Saved PyTorch Model State to model.pth")

Epoch 1
-------------------------------
loss: 1.133569  [   10/  120]
loss: 1.158627  [   40/  120]
loss: 1.093829  [   70/  120]
loss: 1.150908  [  100/  120]
Test Error: 
 Accuracy: 23.3%, Avg loss: 1.119829 

Epoch 2
-------------------------------
loss: 1.141380  [   10/  120]
loss: 1.174284  [   40/  120]
loss: 1.115233  [   70/  120]
loss: 1.137340  [  100/  120]
Test Error: 
 Accuracy: 33.3%, Avg loss: 1.112469 

Epoch 3
-------------------------------
loss: 1.099099  [   10/  120]
loss: 1.133113  [   40/  120]
loss: 1.103652  [   70/  120]
loss: 1.117789  [  100/  120]
Test Error: 
 Accuracy: 33.3%, Avg loss: 1.105561 

Epoch 4
-------------------------------
loss: 1.100623  [   10/  120]
loss: 1.096781  [   40/  120]
loss: 1.119952  [   70/  120]
loss: 1.107019  [  100/  120]
Test Error: 
 Accuracy: 33.3%, Avg loss: 1.099390 

Epoch 5
-------------------------------
loss: 1.117166  [   10/  120]
loss: 1.108220  [   40/  120]
loss: 1.082521  [   70/  120]
loss: 1.101103  [  100

## Evaluating the model using softmax + argmax prediction of the most likely variation (just a little bit more visual)

In [ ]:
model = NeuralNetwork().to(device)
model.load_state_dict(torch.load("./Data/model.pth", weights_only=True))

classes = [
    "Iris-setosa",
    "Iris-versicolor",
    "Iris-virginica"
]

# Rotate throuh i through basically the whole test set

for i in range(len(testdata)):
    total = 0.0
    random_row = np.random.randint(0, len(testdata))
    example = testdata[random_row][0].unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad(): # both of these turn off processes that are only needed for training
        prediction = model(example)
        probabilities = torch.softmax(prediction, dim=1)
        likely_variety = classes[probabilities.argmax()]
        total = total + max(probabilities.flatten().cpu().numpy())
        print(f"Predicted variety: {likely_variety}, Probabilities: {probabilities.cpu().numpy()}")
        print(f"Percentage \"correct\": {total/(1):.2%}")
        i += 1

Predicted variety: Iris-versicolor, Probabilities: [[0.00123357 0.99576783 0.00299856]]
Percentage "correct": 99.58%
Predicted variety: Iris-virginica, Probabilities: [[6.6767200e-07 1.3314757e-03 9.9866784e-01]]
Percentage "correct": 99.87%
Predicted variety: Iris-setosa, Probabilities: [[9.9968803e-01 3.1196023e-04 6.8102338e-13]]
Percentage "correct": 99.97%
Predicted variety: Iris-virginica, Probabilities: [[6.6767200e-07 1.3314757e-03 9.9866784e-01]]
Percentage "correct": 99.87%
Predicted variety: Iris-setosa, Probabilities: [[9.9967229e-01 3.2777520e-04 1.0856925e-12]]
Percentage "correct": 99.97%
Predicted variety: Iris-setosa, Probabilities: [[9.9994814e-01 5.1886120e-05 5.1311409e-15]]
Percentage "correct": 99.99%
Predicted variety: Iris-setosa, Probabilities: [[9.9968803e-01 3.1196023e-04 6.8102338e-13]]
Percentage "correct": 99.97%
Predicted variety: Iris-versicolor, Probabilities: [[7.6177710e-04 9.6294677e-01 3.6291417e-02]]
Percentage "correct": 96.29%
Predicted variety: 

# Summary
This may be overfitted. I am getting results around 99% most of the time using the weights.
Things I could still improve:
- Benchmark the model using different variables
- Visualise the training better using e.g. sns or TensorBoard
- Test the model further using e.g. a larger dataset

I also wanted to present it in a much fancier way but using a Jupyter Notebook ended up being a lot simpler.